**Table of contents**<a id='toc0_'></a>    
- [Title:](#toc1_1_1_)    
    - [Authors:](#toc1_1_2_)    
    - [Abstract:](#toc1_1_3_)    
    - [References](#toc1_1_4_)    
    - [Assumption](#toc1_1_5_)    
    - [Mathematical basis](#toc1_1_6_)    
    - [Numerical method](#toc1_1_7_)    
        - [Step 1) Simulate $X_u|X_t$ using the noncentral $\chi^2$ distribution](#toc1_1_7_1_1_)    
        - [Step 2) $\int_t^u \frac{ds}{X_s}$ Given $X_u, X_t$](#toc1_1_7_1_2_)    
        - [Step 3) Exact method: Simulate $V_T$ with CDF by Laplace transform](#toc1_1_7_1_3_)    
        - [Step 3) Almost exact method: Simulate $V_T$ with assumed distribution](#toc1_1_7_1_4_)    
        - [Finally get the option price](#toc1_1_7_1_5_)    
  - [Case I: Eq. (4.2) in Baldeaux (2012)](#toc1_2_)    
  - [Case II: Set 2 in Kouarfate et al. (2021)](#toc1_3_)    
  - [Case III:in Kouarfate et al. (2021)](#toc1_4_)    
  - [Case IV:](#toc1_5_)    
  - [Pricing with Time Discreteization using Euler/Milstein scheme, Exact Stepping, Almost Exact Stepping](#toc1_6_)    
  - [Pricing with Exact Simulation](#toc1_7_)    
  - [Pricing with IG approximation (Almost Exact Simulation)](#toc1_8_)    
- [Pricing with FFT method](#toc2_)    
- [Pricing with approximation IV](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

### <a id='toc1_1_1_'></a>[Title:](#toc0_)
__Exact simulation and Almost exact simulation of the 3/2 model__

### <a id='toc1_1_2_'></a>[Authors:](#toc0_)
* Jan Baldeaux
* Choi J, Kwok YK
* Medvedev, A., Scaillet, O.

### <a id='toc1_1_3_'></a>[Abstract:](#toc0_)
* Baldeaux J (2012) Exact simulation of the 3/2 model. Int J Theor Appl Finan 15:1250032. https://doi.org/10.1142/S021902491250032X

This paper discusses the exact simulation of the stock price process underlying the 3/2 model. Using a result derived by Craddock and Lennox using Lie Symmetry Analysis, we adapt the Broadie-Kaya algorithm for the simulation of affine processes to the 3/2 model. We also discuss variance reduction techniques and find that conditional Monte Carlo techniques combined with quasi-Monte Carlo point sets result in significant variance reductions.

* Choi J, Kwok YK (2024) Simulation schemes for the Heston model with Poisson conditioning. European Journal of Operational Research 314(1):363–376. https://doi.org/10.1016/j.ejor.2023.10.048

Exact simulation schemes under the Heston stochastic volatility model (e.g., Broadie–Kaya and Glasserman–Kim) suffer from computationally expensive modified Bessel function evaluations. We propose a new exact simulation scheme without the modified Bessel function, based on the observation that the conditional integrated variance can be simplified when conditioned by the Poisson variate used for simulating the terminal variance. Our approach also enhances the low-bias and time discretization schemes, which are suitable for pricing derivatives with frequent monitoring. Extensive numerical tests reveal the good performance of the new simulation schemes in terms of accuracy, efficiency, and reliability when compared with existing methods.

* Medvedev, A., & Scaillet, O. (2007). Approximation and calibration of short-term implied volatilities under jump-diffusion stochastic volatility. The Review of Financial Studies, 20(2), 427-459.https://doi.org/10.1093/rfs/hhl013

We derive an asymptotic expansion formula for option implied volatility under a two factor jump-diffusion stochastic volatility model when time-to-maturity is small. We further propose a simple calibration procedure of an arbitrary parametric model to short-term near-the-money implied volatilities. An important advantage of our approximation is that it is free of the unobserved spot volatility. Therefore, the model can be calibrated on option data pooled across different calendar dates to extract information from the dynamics of the implied volatility smile. An example of calibration to a sample of S&P 500 option prices is provided. 


### <a id='toc1_1_4_'></a>[References](#toc0_)
* Baldeaux J (2012) Exact simulation of the 3/2 model. Int J Theor Appl Finan 15:1250032.
* Kouarfate IR, Kouritzin MA, MacKay A (2021) Explicit Solution Simulation Method for the 3/2 Model. In: Hernández‐Hernández D, Leonardi F, Mena RH, Pardo Millán JC (eds) Advances in Probability and Mathematical Statistics. Springer International Publishing, Cham, pp 123–145
* M. Jeanblanc, M. Yor and M. Chesney, Mathematical Methods for Financial Markets (Springer Finance, Springer, 2009).
* Choi J, Kwok YK (2024) Simulation schemes for the Heston model with Poisson conditioning. European Journal of Operational Research 314(1):363–376.
* Medvedev, A., & Scaillet, O. (2007). Approximation and calibration of short-term implied volatilities under jump-diffusion stochastic volatility. The Review of Financial Studies, 20(2), 427-459.

### <a id='toc1_1_5_'></a>[Assumption](#toc0_)

According to Baldeaux(2013), The dynamics of the stock price under the 3/2 model under the risk-neutral measure are given by

$$
 \frac{dS_t}{S_t} = rdt + \sqrt{V_t}\rho dW_t^1 + \sqrt{V_t}\sqrt{1-\rho^2}dW_t^2 \tag{1}
$$

$$
 \frac{dV_t}{V_t} = \kappa (\theta - V_t)dt + \epsilon \sqrt{V_t}dW_t^1
$$

which is equivalent to

$$
 dV_t = \kappa V_t (\theta - V_t)dt + \epsilon V_t^{3/2}dW_t^1 \tag{2}
$$

where $W_t^1$ and $W_t^2$ are independent Brownian motions. Regarding the parameters, $r$ represents the constant interest rate, $\rho$ the instantaneous correlation between the return on the stock and the variance process and $\epsilon$ governs the volatility of volatility.The speed of mean reversion is given by $\kappa V_t$ and $\theta$ denotes the long-run mean of the variance process.

### <a id='toc1_1_6_'></a>[Mathematical basis](#toc0_)

Defining $X_t = \frac{1}{V_t}$, we obtain

$$
dX_t = (\kappa + \epsilon^2 - \kappa\theta X_t)dt - \epsilon \sqrt{X_t}dW_t^1 \tag{3}
$$
Hence, using the process $X_t$, we obtain the following dynamics for the stock price, where $u > t$

$$
S_u = S_t \exp\lbrace r(u-t) - 1/2 \int_t^u(X_s)^{-1}ds + \rho \int_t^u({\sqrt{X_s})^{-1}dW_s^1}\rbrace \exp \lbrace \sqrt{1-\rho^2} \int_t^u(\sqrt{X_s})^{-1} dW_s^2\rbrace \tag{4}
$$
From Baldeaux(2013), study $\log(X_t)$

$$
\int_t^u({\sqrt{X_s})^{-1}dW_s^1} = \frac{1}{\epsilon} (log(\frac{X_t}{X_u}) + (k + \frac{\epsilon^2}{2})\int_{t}^{u}\frac{ds}{X_s}-k\theta(u-t)) \tag{5}
$$

Therefore, the only thing we need to know is the distribution of $X_t$ and $\int_{t}^{u}\frac{ds}{X_s}$ conditional on $X_t$

### <a id='toc1_1_7_'></a>[Numerical method](#toc0_)
Using Broadie-Kaya algorithm, we specify the simulation as 3 steps

##### <a id='toc1_1_7_1_1_'></a>[Step 1) Simulate $X_u|X_t$ using the noncentral $\chi^2$ distribution](#toc0_)
$X_u$ is distributed as a noncentral $ \chi^2 $ distribution

$$
\frac{X_u{\rm exp}\lbrace \kappa \theta (u-t) \rbrace}{c(u-t)} \sim \chi^2(\delta, \alpha) \tag{6}
$$

where
$$
\delta = \frac{4(\kappa + \epsilon^2)}{\epsilon^2}, \quad \alpha = \frac{X_t}{c(u-t)}, \quad c(t) = \frac{\epsilon^2({\rm exp}\lbrace \kappa\theta u \rbrace - 1)}{4\kappa\theta}
$$

##### <a id='toc1_1_7_1_2_'></a>[Step 2) $\int_t^u \frac{ds}{X_s}$ Given $X_u, X_t$](#toc0_)
We first derive the characteristic function of $\int_u^t \frac{ds}{X_s}$, which is provided in Baldeaux(2013)

$$
E\left({\rm exp}\left\lbrace -a^* \int_0^t \frac{ds}{X_s} \ \bigg| \ X_t \right\rbrace \right) = \frac{I_{\sqrt{\nu^2+8a/\epsilon^2}}\left(-\frac{j\sqrt{X_tX_u}}{{\rm sinh}\left(j\Delta\right)}\right)}{I_{\nu}\left(-\frac{j\sqrt{X_tX_u}}{{\rm sinh}\left(j\Delta\right)}\right)}\tag{7}
$$

where $j=-\frac{2\kappa\theta}{\epsilon^2}$, $\Delta=\frac{u\epsilon^2}{4}-\frac{t\epsilon^2}{4}$, $v=\frac{n}{2}-1$.

##### <a id='toc1_1_7_1_3_'></a>[Step 3) Exact method: Simulate $V_T$ with CDF by Laplace transform](#toc0_)
Use Laplace transform to get the CDF of $\int_t^u \frac{ds}{X_s}$, then get the price.

##### <a id='toc1_1_7_1_4_'></a>[Step 3) Almost exact method: Simulate $V_T$ with assumed distribution](#toc0_)
Then we use the characteristic function to generate moment $M1$, $M2$. For simplicity, we assume that $\int_t^u \frac{ds}{X_s}$ follows the Inverse-Gaussian distribution / Gamma distribution / Log-normal distribution, then we can simulate $\int_t^u \frac{ds}{X_s}$.

##### <a id='toc1_1_7_1_5_'></a>[Finally get the option price](#toc0_)
$$
log(S_u) \sim N(log(S_t)+r(u-t)-\frac{1}{2}\int_t^u \frac{ds}{X_s}+\rho\int_t^u({\sqrt{X_s})^{-1}dW_s^1},  \sigma^2(t,u))\tag{8}
$$
where 
$$
\sigma^2(t, u) = (1-\rho^2)\int_t^u \frac{ds}{X_s}
$$
The option price is $C_{BS}(K,S_u, \sigma(t, u))$

In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
import numpy as np
import time
import scipy.stats as spst
import scipy.special as spsp
import copy 
#import mp math as mp

import sys
sys.path.insert(sys.path.index("")+1, "C:/Users/27261/Desktop/3_Courses in PHBS/3_09_AppliedStochasticProcess/Project_sv32_EMC")
import pyfeng as pf
import pyfeng.ex as pfex
np.set_printoptions(precision=4)
from thesis_demo_utils import *

case_names = case_dict.keys()

## <a id='toc1_6_'></a>[Pricing with Time Discretization using Euler/Milstein scheme, Exact Stepping, Almost Exact Stepping](#toc0_)

In [33]:
# Milstein cannot price Case III and Case VII
run_valuation(pfex.Sv32McTimeStep, 1, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: -0.00025492647847152883 | dt: 0.0002
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 31.359221 秒
------------------------------
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 11.353708 秒


KeyboardInterrupt: 

In [14]:
# Exact Stepping with 1 / NCX2, cannot price Case V and Case VII
run_valuation(pfex.Sv32McTimeStep, 2, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 3.542205 秒


KeyboardInterrupt: 

In [ ]:
# Almost Exact Stepping with Poison-Gamma distribution, cannot price Case II, Case III, Case VI and Case VII
run_valuation(pfex.Sv32McTimeStep, 3, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: -0.0004362191254071446 | dt: 0.002
[Case I] 运行耗时: 5.693315 秒
------------------------------
Bias: [-0.0135 -0.0121 -0.0103] | dt: 0.0002
[Case II] 运行耗时: 28.617692 秒
------------------------------
Bias: [-0.0197 -0.0188 -0.017 ] | dt: 0.002
[Case III] 运行耗时: 2.876173 秒
------------------------------
Bias: [0.0031 0.0706 0.0035 0.0033 0.0031] | dt: 0.002
[Case IV] 运行耗时: 0.605714 秒
------------------------------
Bias: [0.0085 0.0085 0.0068] | dt: 0.002
[Case V] 运行耗时: 0.594140 秒
------------------------------
Bias: [0.1214 0.121  0.121  0.1211 0.1207] | dt: 2e-05
[Case VI] 运行耗时: 559.179105 秒
------------------------------
Bias: [-0.0192 -0.0176 -0.0161] | dt: 0.002
[Case VII] 运行耗时: 2.887707 秒
------------------------------


In [ ]:
# QE method, cannot price Case VI
run_valuation(pfex.Sv32McTimeStep, 4, case_names, case_dict)


========== 正在运行模型: Sv32McTimeStep ==========
Bias: -0.00014108583206956515 | dt: 0.0002
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 68.663679 秒
------------------------------
Bias: [-0.0136 -0.0164 -0.0168] | dt: 0.0002
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 35.966439 秒
------------------------------
Bias: [-0.0202 -0.0228 -0.0207] | dt: 0.0002
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 38.544705 秒
------------------------------
Bias: [-7.7072e-04  6.6927e-02  1.3558e-05  8.2636e-05  6.5103e-05] | dt: 0.0002
[Case IV]: Near expiration and atm 
 运行耗时: 10.547259 秒
------------------------------
Bias: [0.0006 0.0007 0.0009] | dt: 0.0002
[Case V]: Near expiration only 
 运行耗时: 10.777851 秒
------------------------------
Bias: [3.0574e+21 3.0574e+21 3.0574e+21 3.0574e+21 3.0574e+21] | dt: 0.0002
[Case VI]: atm only 
 运行耗时: 83.574298 秒
------------------------------
Bias: [-0.0197 -0.0216 -0.0198] | dt: 0.0002
[Case VII]: Lewis AL (2000) Option valuation und

## <a id='toc1_7_'></a>[Pricing with Exact Simulation](#toc0_)
## Exact simulation **cannot** price case II to case VII

In [ ]:
run_valuation(pfex.Sv32McBaldeaux2012Exact, None, case_names, case_dict)


========== 正在运行模型: Sv32McBaldeaux2012Exact ==========


## <a id='toc1_8_'></a>[Pricing with IG approximation (Almost Exact Simulation)](#toc0_)
## Almost exact simulation **cannot** price case II to V, and case VII

In [75]:
run_valuation(pfex.Sv32McChoiKwok2023Ig, None, case_names, case_dict)


========== 正在运行模型: Sv32McChoiKwok2023Ig ==========
Bias: -0.006304993756619082 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012) 
 运行耗时: 1.411961 秒
------------------------------
Bias: [0.1918 0.2362 0.2574] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021) 
 运行耗时: 1.712543 秒
------------------------------
Bias: [0.0284 0.0318 0.0349] | dt: None
[Case III]: in Kouarfate et al. (2021) 
 运行耗时: 1.357137 秒
------------------------------
Bias: [-5.5587 -5.2872 -5.1345 -4.903  -4.6638] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 2.834469 秒
------------------------------
Bias: [-5.2551 -5.2586 -6.3558] | dt: None
[Case V]: Near expiration only 
 运行耗时: 2.775194 秒
------------------------------
Bias: [-0.0032  0.0003  0.0041  0.0078  0.0108] | dt: None
[Case VI]: atm only 
 运行耗时: 1.352041 秒
------------------------------
Bias: [0.0289 0.033  0.0358] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: with Mathematica code. Finance Press 
 运行耗时: 1

# <a id='toc2_'></a>[Pricing with FFT method](#toc0_)
## FFT method **cannot** price case IV, case V, case VI

In [37]:
run_valuation(pf.sv_fft.Sv32Fft, None, case_names, case_dict)


========== 正在运行模型: Sv32Fft ==========
Bias: -2.434335364398521e-07 | dt: None
[Case I]: Eq. (4.2) in Baldeaux (2012), ATM 
 运行耗时: 0.050365 秒
------------------------------
Bias: [-0.0005 -0.0001 -0.001 ] | dt: None
[Case II]: Set 2 in Kouarfate et al. (2021), ATM 
 运行耗时: 0.040167 秒
------------------------------
Bias: [-0.0005 -0.0012 -0.0009] | dt: None
[Case III]: in Kouarfate et al. (2021), ATM 
 运行耗时: 0.038366 秒
------------------------------
Bias: [8.1971e+58 1.3137e+59 1.6244e+59 1.7718e+59 1.7835e+59] | dt: None
[Case IV]: Near expiration and atm 
 运行耗时: 0.037026 秒
------------------------------
Bias: [ 6.3965e+59 -2.6550e+58 -5.9570e+59] | dt: None
[Case V]: Near expiration only 
 运行耗时: 0.033807 秒
------------------------------
Bias: [0.0754 0.0751 0.0753 0.0755 0.0752] | dt: None
[Case VI]: atm only 
 运行耗时: 0.035222 秒
------------------------------
Bias: [-2.1799e-05 -4.5977e-05  4.8946e-05] | dt: None
[Case VII]: Lewis AL (2000) Option valuation under stochastic volatility: 